In [1]:
print("here is where we discover our data, and get sone insights")

here is where we discover our data, and get sone insights


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
used = pd.read_csv("/content/used_cars.csv")
used.head(1)

,page,car_type,car_name,year,mileage,transmission,fuel,price,location,brand,model,car_url,image_url,scraped_at
0,1,used,Mercedes C 180 2025,2025,"5,000 KM",Automatic,Gas,"3,300,000 EGP","Tagamo3 - New Cairo, Cairo",Mercedes,C 180,https://eg.hatla2ee.com/en/car/mercedes/c-180/...,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 14:20:32




*   drop page num --> done
*   you may drop car name, as we got the same data in other columns

*   transform price & mileage into numeric column --> done
*   get more data from location, like city, town. --> done


*   drop car_URL --> done
*   we may keep scrapped at as an indicator for starting the new job.


*   fix the nulls in model column --> done

*   extracted the governrate and fixed the nulls in it.
*   











In [7]:
used.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10173 entries, 0 to 10172
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   page          10173 non-null  int64 
 1   car_type      10173 non-null  object
 2   car_name      10173 non-null  object
 3   year          10173 non-null  int64 
 4   mileage       10173 non-null  object
 5   transmission  10173 non-null  object
 6   fuel          10173 non-null  object
 7   price         10173 non-null  object
 8   location      10173 non-null  object
 9   brand         10173 non-null  object
 10  model         10171 non-null  object
 11  car_url       10173 non-null  object
 12  image_url     10173 non-null  object
 13  scraped_at    10173 non-null  object
dtypes: int64(2), object(12)
memory usage: 1.1+ MB




*   there are 2 nulls in Model column --> to be fixed



In [10]:
used[used['model'].isna()]

,page,car_type,car_name,year,mileage,transmission,fuel,price,location,brand,model,car_url,image_url,scraped_at
4577,235,used,Nissan Sunny 2015,2015,"185,000 KM",Automatic,Gas,"490,000 EGP",Nissan,Sunny,NaN,https://eg.hatla2ee.com/en/car/nissan/sunny/71...,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 15:36:38
5886,322,used,BMW X6 2019,2019,"90,000 KM",Automatic,Gas,"2,800,000 EGP",BMW,X6,NaN,https://eg.hatla2ee.com/en/car/bmw/x6/7197351,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 16:02:21


In [11]:
# fixing the nulls in model column

used.loc[4577 , 'model'] = "Sunny"
used.loc[4577 , 'brand'] = "Nissan"
used.loc[4577 , 'location'] = ""


In [12]:
used[used['model'].isna()]

,page,car_type,car_name,year,mileage,transmission,fuel,price,location,brand,model,car_url,image_url,scraped_at
5886,322,used,BMW X6 2019,2019,"90,000 KM",Automatic,Gas,"2,800,000 EGP",BMW,X6,NaN,https://eg.hatla2ee.com/en/car/bmw/x6/7197351,https://legion-images.hatla2ee.com/listing_ima...,2026-04-30 16:02:21


In [13]:
used.loc[5886 , 'model'] = "X6"
used.loc[4577 , 'brand'] = "BMW"
used.loc[4577 , 'location'] = ""

In [15]:
used.drop(columns = ["page", "car_url", "image_url"] , inplace = True)

In [16]:
used.head(2)

,car_type,car_name,year,mileage,transmission,fuel,price,location,brand,model,scraped_at
0,used,Mercedes C 180 2025,2025,"5,000 KM",Automatic,Gas,"3,300,000 EGP","Tagamo3 - New Cairo, Cairo",Mercedes,C 180,2026-04-30 14:20:32
1,used,BMW X5 M 2015,2015,"180,000 KM",Automatic,Gas,"1,750,000 EGP","Sheikh Zayed City, Giza",BMW,X5 M,2026-04-30 14:20:34


In [17]:
# transform the columns "price" & "mileage"

In [18]:
used["price"] = pd.to_numeric(used["price"].str.replace(r"\D", "", regex=True))
used["mileage"] = pd.to_numeric(used["mileage"].str.replace(r"\D", "", regex=True))

In [22]:
#used.info()

In [25]:
# dealing with location
used["location"]

,location
0,"Tagamo3 - New Cairo, Cairo"
1,"Sheikh Zayed City, Giza"
2,"Heliopolis, Cairo"
3,"Sheikh Zayed City, Giza"
4,"Sheikh Zayed City, Giza"
...,...
10168,Cairo
10169,Cairo
10170,"Obour City, Cairo"
10171,Cairo


In [67]:
import re
import pandas as pd

gov_aliases = {
    "cairo": ["cairo"],
    "giza": ["giza", "6th of october", "october", "sheikh zayed"],
    "alex": ["alex", "alexandria"],
    "dakahlia": ["dakahlia", "mansoura"],
    "red sea": ["red sea", "hurghada"],
    "beheira": ["beheira", "damanhur"],
    "fayoum": ["fayoum"],
    "gharbia": ["gharbia", "tanta"],
    "ismailia": ["ismailia"],
    "monufia": ["monufia", "shebin el kom"],
    "minya": ["minya"],
    "qalyubia": ["qalyubia", "benha"],
    "new valley": ["new valley"],
    "suez": ["suez"],
    "aswan": ["aswan"],
    "assiut": ["assiut"],
    "beni suef": ["beni suef"],
    "port said": ["port said"],
    "damietta": ["damietta"],
    "sharkia": ["sharkia", "zagazig"],
    "south sinai": ["south sinai", "sharm"],
    "kafr el sheikh": ["kafr el sheikh"],
    "matrouh": ["matrouh"],
    "luxor": ["luxor"],
    "qena": ["qena"],
    "north sinai": ["north sinai"],
    "sohag": ["sohag"]
}


In [68]:
def merge_aliases(base, new):
    for key, values in new.items():
        if key in base:
            base[key] = list(set(base[key] + values))  # merge + deduplicate
        else:
            base[key] = values
    return base

In [72]:
new_aliases = {
    "assiut": ["assyut", "asyut", "assiut"],
    "sharkia": ["sharqia", "sharkia", "faqous", "bilbeis", "abu kabir", "dyarb negm"],
    "fayoum": ["faiyum", "fayoum", "el faiyum"],
    "beni suef": ["beni suef", "ben suef"],
    "new valley": ["el wadi el gedid", "wadi gedid", "kharga", "farafra"],
    "aswan": ["kom ombo"],
    "matrouh": ["mariotya"],
    "sinai": ["sinai"],
    "monufia": ["menofia", "ashmun", "birkat as sab" , "Monufia" ,"Shibin el Kom" , "El Bagour", "Tala" , "Ashmoun" , "Quwaysna"]
}

gov_aliases = merge_aliases(gov_aliases, new_aliases)

In [34]:
used.head(2)

,car_type,car_name,year,mileage,transmission,fuel,price,location,brand,model,scraped_at,city
0,used,Mercedes C 180 2025,2025,5000,Automatic,Gas,3300000,"Tagamo3 - New Cairo, Cairo",Mercedes,C 180,2026-04-30 14:20:32,cairo
1,used,BMW X5 M 2015,2015,180000,Automatic,Gas,1750000,"Sheikh Zayed City, Giza",BMW,X5 M,2026-04-30 14:20:34,giza


In [36]:
#used.info()

In [73]:
#used.loc[used["city"].isna(), "location"].unique()

In [74]:
def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

In [75]:
def extract_governorate_and_city(location):
    if pd.isna(location) or str(location).strip() == "":
        return None, None

    loc = normalize(location)

    best_match = None
    best_gov = None

    # prioritize longest match
    for gov, aliases in gov_aliases.items():
        for alias in sorted(aliases, key=len, reverse=True):
            if alias in loc:
                best_match = alias
                best_gov = gov
                break
        if best_match:
            break

    if best_match:
        city = loc.replace(best_match, "").strip()
        return best_gov, (city if city else None)

    # fallback → treat whole thing as city
    return None, loc

In [76]:
used[["governorate", "city"]] = used["location"].apply(
    lambda x: pd.Series(extract_governorate_and_city(x))
)

In [77]:
#used.head(2)

In [78]:
#used.info()

In [79]:
used.loc[used["governorate"].isna(), "location"].unique()

array(['', 'BMW', '5th Settlement'], dtype=object)